# Instructor solution — Unit 3 lab

One reference implementation. Because training uses random exploration, students' routes and learning curves will vary.

In [ ]:
import random

SIZE, START, GOAL = 4, (0, 0), (3, 3)
ACTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]
BLOCKED = {(1, 1), (2, 1)}

def step_with_obstacles(state, action):
    row, col = state
    dr, dc = action
    candidate = (max(0, min(SIZE - 1, row + dr)), max(0, min(SIZE - 1, col + dc)))
    next_state = state if candidate in BLOCKED else candidate
    done = next_state == GOAL
    return next_state, 10 if done else -1, done

q = {(r, c): [0.0] * len(ACTIONS) for r in range(SIZE) for c in range(SIZE)}
alpha, gamma, epsilon = 0.2, 0.95, 0.2
for episode in range(1500):
    state = START
    for _ in range(40):
        if random.random() < epsilon:
            action_index = random.randrange(len(ACTIONS))
        else:
            action_index = max(range(len(ACTIONS)), key=lambda i: q[state][i])
        next_state, reward, done = step_with_obstacles(state, ACTIONS[action_index])
        target = reward if done else reward + gamma * max(q[next_state])
        q[state][action_index] += alpha * (target - q[state][action_index])
        state = next_state
        if done:
            break

state, route = START, [START]
for _ in range(20):
    if state == GOAL:
        break
    action_index = max(range(len(ACTIONS)), key=lambda i: q[state][i])
    state, _, _ = step_with_obstacles(state, ACTIONS[action_index])
    route.append(state)
print(route)
assert route[-1] == GOAL
assert not (set(route) & BLOCKED)

The next-state candidate is clamped to the grid; if it is blocked, the agent stays put and receives the normal step penalty. An epsilon of zero from the beginning can trap a tie-breaking greedy policy in uninformative behavior before it discovers the goal. A more negative step reward increases pressure to find a shorter path.